# The Efficiency of Joy: Steam Market Value Analysis

### Project Goal:
Analyze the relationship between game price (INR) and player satisfaction across AAA and Indie/AA titles to determine the 'Return on Investment' (ROI) for consumers.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual settings
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load the Data
We are using the `steam_top_games_2026.csv` dataset.

In [ ]:
df = pd.read_csv('steam_top_games_2026.csv')
df.head()

## 2. Data Cleaning & Tier Categorization
We categorize games into 'AAA' and 'Indie/AA' based on their publisher names and price points to ensure accuracy.

In [ ]:
# Define major AAA publishers
aaa_publishers = [
    "Electronic Arts", "EA ", "Ubisoft", "CAPCOM", "Xbox Game Studios", "Bethesda", "SEGA", 
    "Activision", "2K", "Warner Bros", "WB Games", "Rockstar", "Square Enix", 
    "PlayStation Publishing", "BANDAI NAMCO", "Sony", "Nintendo", "Microsoft", 
    "Deep Silver", "THQ Nordic", "Paradox", "Larian Studios", "CD PROJEKT RED", 
    "FromSoftware", "Konami", "Take-Two", "Blizzard", "KRAFTON", "Bungie"
]

def categorize_game(row):
    if row['price_usd'] >= 40:
        return 'AAA'
    pub = str(row['publisher']).lower()
    for aaa_pub in aaa_publishers:
        if aaa_pub.lower() in pub:
            return 'AAA'
    return 'Indie/AA'

# Filter out zero-review games for statistical significance
df = df[(df['positive_reviews'] + df['negative_reviews']) > 500].copy()

# Apply categorization
df['category'] = df.apply(categorize_game, axis=1)
print("Updated Category Counts:")
print(df['category'].value_counts())

## 3. Custom Metrics & Currency Conversion
Calculating User Rating %, INR price, and the Joy Efficiency Score (JES).

In [ ]:
EXCHANGE_RATE = 83

# Convert to INR
df['price_inr'] = df['price_usd'] * EXCHANGE_RATE

# Satisfaction Rating
df['user_rating_pct'] = (df['positive_reviews'] / (df['positive_reviews'] + df['negative_reviews'])) * 100

# Joy Efficiency Score (JES)
# Formula: Rating / (Price + 1)
df['jes'] = df['user_rating_pct'] / (df['price_inr'] + 1)

# Normalized metric for AAA comparison
df['joy_per_1000_inr'] = (df['user_rating_pct'] / (df['price_inr'] + 1)) * 1000

df[['name', 'price_inr', 'user_rating_pct', 'jes']].sort_values(by='jes', ascending=False).head()

## 4. Visualizations

In [ ]:
# 4.1 AAA vs Indie: Efficiency Gap (Log Scale)
plt.figure(figsize=(10, 6))
sns.barplot(x='category', y='jes', data=df, palette='viridis', errorbar=None)
plt.yscale('log')
plt.title('Joy Efficiency Score Comparison (Log Scale)')
plt.ylabel('Average JES (Log Scale)')
plt.show()

In [ ]:
# 4.2 Top 15 AAA Games: Joy per 1,000 INR Spent
aaa_only = df[df['category'] == 'AAA'].sort_values(by='joy_per_1000_inr', ascending=False).head(15)
plt.figure(figsize=(12, 7))
sns.barplot(x='joy_per_1000_inr', y='name', data=aaa_only, palette='magma')
plt.title('Top 15 AAA Games by Normalized Value')
plt.xlabel('Joy Units (per 1,000 INR)')
plt.show()

In [ ]:
# 4.3 User Rating Distribution Boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(x='category', y='user_rating_pct', data=df, palette='Set2')
plt.title('User Rating Distribution: AAA vs Indie/AA')
plt.show()

In [ ]:
# 4.4 The Value Sweet Spot (Price vs Rating)
sweet_spot = df[(df['price_inr'] < 3000) & (df['user_rating_pct'] > 70)]
plt.figure(figsize=(12, 6))
sns.scatterplot(x='price_inr', y='user_rating_pct', size='jes', hue='category', 
                data=sweet_spot, palette={'AAA': 'red', 'Indie/AA': 'blue'}, alpha=0.6)
plt.title('The "Value Sweet Spot": Games < ₹3000 with >70% Rating')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 5. Conclusion
- **Efficiency Lead**: Indie games dominate the 'Joy per Rupee' metrics.
- **Quality Floors**: AAA games are consistent but suffer from diminishing returns due to high price points.
- **The Sweet Spot**: High-value gaming is concentrated in the ₹500 - ₹1,500 range for Indie/AA titles.